<a href="https://colab.research.google.com/github/asmaa-2003/LSTM-Anomaly--detection/blob/main/xgboost_fault_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
"""
MetroPT1 - Predictive Maintenance (REAL)
========================================
الهدف: التنبؤ بحدوث عطل بعد فترة زمنية (مثلاً 30 دقيقة)
"""

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────
# 1. تحميل البيانات
# ─────────────────────────────────────
def load_data(path):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    print("Shape:", df.shape)
    return df


# ─────────────────────────────────────
# 2. Feature Engineering (بدون تسريب)
# ─────────────────────────────────────
def create_features(df):
    df = df.copy()

    sensor_cols = [
        "TP2","TP3","H1","DV_pressure","Reservoirs",
        "Oil_temperature","Flowmeter","Motor_current"
    ]

    for col in sensor_cols:
        if col in df.columns:
            df[f"{col}_mean"] = df[col].rolling(20).mean()
            df[f"{col}_std"]  = df[col].rolling(20).std()
            df[f"{col}_diff"] = df[col].diff()

    df = df.fillna(0)
    return df


# ─────────────────────────────────────
# 3. Create REAL target (future fault)
# ─────────────────────────────────────
def create_target(df, horizon=1800):
    """
    horizon = عدد الثواني (1800 = 30 دقيقة)
    """
    df = df.copy()

    # تعريف بسيط للعطل (يمكن تحسينه)
    fault_now = (
        (df["TP3"] < 8.5) |
        (df["Oil_temperature"] > 75) |
        (df["Motor_current"] > 5)
    ).astype(int)

    # الهدف: هل سيحدث عطل في المستقبل؟
    df["fault_future"] = fault_now.shift(-horizon)

    df = df.dropna()
    df["fault_future"] = df["fault_future"].astype(int)

    print("Fault rate:", df["fault_future"].mean())

    return df


# ─────────────────────────────────────
# 4. Split (زمني)
# ─────────────────────────────────────
def split_data(df):
    drop_cols = ["timestamp", "fault_future"]
    X = df.drop(columns=drop_cols)
    y = df["fault_future"]

    split = int(len(df)*0.8)

    X_train = X.iloc[:split]
    X_test  = X.iloc[split:]
    y_train = y.iloc[:split]
    y_test  = y.iloc[split:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test


# ─────────────────────────────────────
# 5. Model
# ─────────────────────────────────────
def train_model(X_train, y_train):

    scale_pos = (y_train==0).sum() / max((y_train==1).sum(),1)

    model = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
    )

    model.fit(X_train, y_train)

    return model


# ─────────────────────────────────────
# 6. Evaluation (with threshold tuning)
# ─────────────────────────────────────
def evaluate(model, X_test, y_test):

    proba = model.predict_proba(X_test)[:,1]

    # تجربة عدة thresholds
    best_f1 = 0
    best_t  = 0.5

    for t in np.linspace(0.05,0.5,10):
        pred = (proba > t).astype(int)
        f1 = f1_score(y_test, pred)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    print("Best threshold:", best_t)

    y_pred = (proba > best_t).astype(int)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    try:
        auc = roc_auc_score(y_test, proba)
        print("AUC:", auc)
    except:
        print("AUC: undefined")

    print("F1:", f1_score(y_test, y_pred))


# ─────────────────────────────────────
# MAIN
# ─────────────────────────────────────
if __name__ == "__main__":

    PATH = "MetroPT1.csv"   # ضع المسار

    df = load_data(PATH)

    df = create_features(df)

    df = create_target(df, horizon=1800)  # 30 دقيقة

    X_train, X_test, y_train, y_test = split_data(df)

    model = train_model(X_train, y_train)

    evaluate(model, X_test, y_test)

Shape: (751182, 21)
Fault rate: 0.3525811935701685
Best threshold: 0.15000000000000002

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.50      0.61    100107
           1       0.42      0.73      0.53     49770

    accuracy                           0.57    149877
   macro avg       0.60      0.61      0.57    149877
weighted avg       0.66      0.57      0.58    149877

AUC: 0.6711716406984811
F1: 0.5302567035480962
